# PASUL 2 — Construire tier-uri progresive (cumulativ, lesion-level)

**Scop:** Pornind de la `train_pool.csv` (cele ~20,300 imagini de antrenare din PASUL 1), construim 5 tier-uri progresive de antrenare:

| Tier | Compoziție | Imagini estimate |
|------|-----------|------------------|
| T1 | doar HAM10000 | ~8,000 |
| T2 | T1 + 25% din restul (BCN/MSK/other) | ~11,000 |
| T3 | T1 + 50% din restul | ~14,000 |
| T4 | T1 + 75% din restul | ~17,000 |
| T5 | tot train_pool | 20,299 |

**Decizii cheie:**

1. **Cumulativ:** T1 ⊆ T2 ⊆ T3 ⊆ T4 ⊆ T5. Garantăm că trecerea de la T_n la T_{n+1} **doar adaugă** date, nu schimbă datele existente.
2. **Lesion-level sample:** sampleul random de 25/50/75% din non-HAM se face la nivel de leziune, nu de imagine. Asta păstrează garanția anti-leakage din PASUL 1 și în interiorul tier-urilor.
3. **Stratificare pe clasă în sample:** când luăm 25% din non-HAM, păstrăm distribuția claselor (DF nu dispare doar pentru că e minoritar).
4. **Random seed fix:** asigurăm reproducibilitate exactă a tier-urilor.

**Output:**
- `tier1_train.csv` ... `tier5_train.csv` — câte un CSV per tier
- `step2_report.md` — raport cu compoziția exactă a fiecărui tier

**Important:** val_fixed.csv și test_fixed.csv nu se modifică. Sunt aceleași pentru toate tier-urile.

---

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
IN_COLAB = True

Mounted at /content/drive


In [2]:
BASE_DIR = '/content/drive/MyDrive/Thesis'
TRAIN_POOL_PATH = f'{BASE_DIR}/pipeline_outputs/step1/train_pool.csv'
OUTPUT_DIR = f'{BASE_DIR}/pipeline_outputs/step2'

TIER_PERCENTAGES = [0.0, 0.25, 0.50, 0.75, 1.0]  # T1=0% non-HAM, T2=25%, ..., T5=100%
ANCHOR_SOURCE = 'HAM10000'  # Sursa care formează nucleul T1
RANDOM_SEED = 42

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input: {TRAIN_POOL_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Tier percentages (non-HAM): {TIER_PERCENTAGES}")
print(f"Anchor source (T1): {ANCHOR_SOURCE}")

Input: /content/drive/MyDrive/Thesis/pipeline_outputs/step1/train_pool.csv
Output: /content/drive/MyDrive/Thesis/pipeline_outputs/step2
Tier percentages (non-HAM): [0.0, 0.25, 0.5, 0.75, 1.0]
Anchor source (T1): HAM10000


## 2. Imports

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
np.random.seed(RANDOM_SEED)

## 3. Citire train_pool.csv

In [4]:
train_pool = pd.read_csv(TRAIN_POOL_PATH)
print(f"Total imagini în train_pool: {len(train_pool):,}")
print(f"Lesion_id-uri unice: {train_pool['lesion_id'].nunique():,}")
print(f"\nDistribuție surse:")
print(train_pool['source'].value_counts().to_string())

Total imagini în train_pool: 20,299
Lesion_id-uri unice: 11,144

Distribuție surse:
source
BCN_20000    9969
HAM10000     8042
other        1637
MSK           651


## 4. Separare anchor (HAM10000) vs non-anchor

Pentru construcția tier-urilor cumulativ, separăm pool-ul în două:

- **Anchor pool:** toate leziunile HAM10000 (apar în toate tier-urile)
- **Non-anchor pool:** restul (BCN, MSK, other) — adăugăm progresiv 25/50/75/100% din ele

In [5]:
# Anchor: HAM10000 (apare 100% în toate tier-urile)
anchor_df = train_pool[train_pool['source'] == ANCHOR_SOURCE].copy()
anchor_lesions = anchor_df['lesion_id'].unique()

# Non-anchor: restul surselor (BCN, MSK, other)
non_anchor_df = train_pool[train_pool['source'] != ANCHOR_SOURCE].copy()
non_anchor_lesions = non_anchor_df['lesion_id'].unique()

print(f"ANCHOR ({ANCHOR_SOURCE}):")
print(f"  Imagini: {len(anchor_df):,}")
print(f"  Leziuni: {len(anchor_lesions):,}")
print(f"\nNON-ANCHOR (rest):")
print(f"  Imagini: {len(non_anchor_df):,}")
print(f"  Leziuni: {len(non_anchor_lesions):,}")
print(f"  Surse: {sorted(non_anchor_df['source'].unique().tolist())}")

ANCHOR (HAM10000):
  Imagini: 8,042
  Leziuni: 5,987

NON-ANCHOR (rest):
  Imagini: 12,257
  Leziuni: 5,157
  Surse: ['BCN_20000', 'MSK', 'other']


## 5. Construire ordine cumulativă pentru leziunile non-anchor

**Idee cheie:** pentru a avea tier-uri cumulative (T2 ⊇ T1, T3 ⊇ T2, etc.), generăm o singură ordonare aleatoare a leziunilor non-anchor. Apoi:

- T2 = anchor + primele 25% din această ordonare
- T3 = anchor + primele 50% din această ordonare  
- T4 = anchor + primele 75% din această ordonare
- T5 = anchor + 100% din această ordonare

Astfel, leziunile incluse în T2 sunt automat și în T3, T4, T5.

**Sub-decizie:** ordonarea e stratificată pe clasă, ca să garantăm că DF (113 leziuni totale, dar majoritar non-HAM) e reprezentat în fiecare tier.

In [6]:
non_anchor_lesion_df = non_anchor_df.groupby('lesion_id').agg(
    dx=('dx', 'first'),
    source=('source', 'first'),
    n_images=('image_id', 'count')
).reset_index()

print(f"Leziuni non-anchor: {len(non_anchor_lesion_df):,}")
print(f"\nDistribuție clase pe leziuni non-anchor:")
print(non_anchor_lesion_df['dx'].value_counts().to_string())

Leziuni non-anchor: 5,157

Distribuție clase pe leziuni non-anchor:
dx
nv       2577
mel       844
bcc       797
bkl       592
akiec     284
df         32
vasc       31


In [7]:
def stratified_cumulative_ordering(lesion_df, stratify_col='dx', seed=RANDOM_SEED):
    """
    Generează o ordonare aleatoare a leziunilor, stratificată pe clasă.

    Pentru fiecare clasă, leziunile sunt amestecate aleator. Apoi clasele
    sunt intercalate astfel încât oricare prefix de mărime k să conțină
    aproximativ proporția corectă din fiecare clasă.

    Asta garantează că primele 25% din ordonare conțin ~25% din DF,
    ~25% din NV, etc. — distribuția e păstrată în orice prefix.
    """
    rng = np.random.RandomState(seed)

    # Pentru fiecare clasă, amestecăm leziunile
    class_orderings = {}
    for cls in lesion_df[stratify_col].unique():
        class_lesions = lesion_df[lesion_df[stratify_col] == cls]['lesion_id'].values
        shuffled = class_lesions.copy()
        rng.shuffle(shuffled)
        class_orderings[cls] = shuffled.tolist()

    # Intercalăm clasele proporțional cu mărimile lor
    # Folosim o tehnică simpla: la fiecare pas alegem clasa cu cel mai mare
    # "deficit" relativ la proporția ei finală
    total = len(lesion_df)
    class_targets = {cls: len(class_orderings[cls]) for cls in class_orderings}
    class_taken = {cls: 0 for cls in class_orderings}

    final_ordering = []
    for step in range(total):
        # Calculăm deficitul fiecărei clase: cât ar trebui să aibă vs cât are
        progress = (step + 1) / total
        deficits = {}
        for cls in class_orderings:
            target_at_this_point = progress * class_targets[cls]
            deficits[cls] = target_at_this_point - class_taken[cls]

        # Alegem clasa cu cel mai mare deficit, dintre cele care mai au leziuni
        eligible = {cls: d for cls, d in deficits.items()
                    if class_taken[cls] < class_targets[cls]}
        chosen = max(eligible, key=eligible.get)

        # Luăm prima leziune din clasa aleasă
        lesion = class_orderings[chosen][class_taken[chosen]]
        final_ordering.append(lesion)
        class_taken[chosen] += 1

    return final_ordering


# Generăm ordonarea
non_anchor_ordering = stratified_cumulative_ordering(non_anchor_lesion_df)
print(f"Ordonare generată: {len(non_anchor_ordering):,} leziuni")
print(f"Primele 5: {non_anchor_ordering[:5]}")
print(f"Ultimele 5: {non_anchor_ordering[-5:]}")

Ordonare generată: 5,157 leziuni
Primele 5: ['SYNTH_ISIC_0000423_downsampled', 'MSK4_0010966', 'SYNTH_ISIC_0006982_downsampled', 'BCN_0001490', 'SYNTH_ISIC_0014770_downsampled']
Ultimele 5: ['BCN_0002987', 'BCN_0000143', 'SYNTH_ISIC_0002976_downsampled', 'BCN_0002266', 'BCN_0005337']


In [8]:
# Verificare: la 25%, 50%, 75%, distribuția claselor e proporțională?
print("Verificare distribuții la fiecare prag (proporție din total per clasă):\n")

class_to_lesion_ids = {}
for cls in non_anchor_lesion_df['dx'].unique():
    class_to_lesion_ids[cls] = set(non_anchor_lesion_df[non_anchor_lesion_df['dx'] == cls]['lesion_id'])

for pct in [0.25, 0.50, 0.75]:
    n = int(round(pct * len(non_anchor_ordering)))
    prefix = set(non_anchor_ordering[:n])
    print(f"La {int(pct*100)}% ({n:,} leziuni):")
    for cls in sorted(class_to_lesion_ids.keys()):
        cls_total = len(class_to_lesion_ids[cls])
        cls_in_prefix = len(prefix & class_to_lesion_ids[cls])
        actual_pct = 100 * cls_in_prefix / cls_total if cls_total > 0 else 0
        print(f"  {cls:8s}: {cls_in_prefix:5,d}/{cls_total:5,d}  ({actual_pct:5.1f}% — target {pct*100:.0f}%)")
    print()

Verificare distribuții la fiecare prag (proporție din total per clasă):

La 25% (1,289 leziuni):
  akiec   :    71/  284  ( 25.0% — target 25%)
  bcc     :   199/  797  ( 25.0% — target 25%)
  bkl     :   148/  592  ( 25.0% — target 25%)
  df      :     8/   32  ( 25.0% — target 25%)
  mel     :   211/  844  ( 25.0% — target 25%)
  nv      :   644/2,577  ( 25.0% — target 25%)
  vasc    :     8/   31  ( 25.8% — target 25%)

La 50% (2,578 leziuni):
  akiec   :   142/  284  ( 50.0% — target 50%)
  bcc     :   398/  797  ( 49.9% — target 50%)
  bkl     :   296/  592  ( 50.0% — target 50%)
  df      :    16/   32  ( 50.0% — target 50%)
  mel     :   422/  844  ( 50.0% — target 50%)
  nv      : 1,288/2,577  ( 50.0% — target 50%)
  vasc    :    16/   31  ( 51.6% — target 50%)

La 75% (3,868 leziuni):
  akiec   :   213/  284  ( 75.0% — target 75%)
  bcc     :   598/  797  ( 75.0% — target 75%)
  bkl     :   444/  592  ( 75.0% — target 75%)
  df      :    24/   32  ( 75.0% — target 75%)
  mel  

## 6. Construire tier-uri (cumulativ)

In [9]:
def build_tier(tier_idx, pct_non_anchor, anchor_df, non_anchor_df, non_anchor_ordering):
    """
    Construiește un tier prin combinarea:
    - Tot anchor_df (leziunile HAM10000)
    - Primele (pct_non_anchor * 100)% din non_anchor_ordering, expandate la imagini
    """
    n_non_anchor_lesions = int(round(pct_non_anchor * len(non_anchor_ordering)))
    selected_non_anchor_lesions = set(non_anchor_ordering[:n_non_anchor_lesions])

    # Imaginile non-anchor selectate
    selected_non_anchor_df = non_anchor_df[
        non_anchor_df['lesion_id'].isin(selected_non_anchor_lesions)
    ].copy()

    # Concatenăm cu anchor
    tier_df = pd.concat([anchor_df, selected_non_anchor_df], ignore_index=True)
    tier_df['tier'] = tier_idx

    return tier_df


# Construire toate cele 5 tier-uri
tiers = {}
for i, pct in enumerate(TIER_PERCENTAGES, start=1):
    tier_df = build_tier(i, pct, anchor_df, non_anchor_df, non_anchor_ordering)
    tiers[i] = tier_df
    print(f"Tier {i} ({int(pct*100)}% non-anchor): {len(tier_df):>6,} imagini, "
          f"{tier_df['lesion_id'].nunique():>6,} leziuni")

Tier 1 (0% non-anchor):  8,042 imagini,  5,987 leziuni
Tier 2 (25% non-anchor): 10,996 imagini,  7,276 leziuni
Tier 3 (50% non-anchor): 14,061 imagini,  8,565 leziuni
Tier 4 (75% non-anchor): 17,300 imagini,  9,855 leziuni
Tier 5 (100% non-anchor): 20,299 imagini, 11,144 leziuni


## 7. Sanity checks: garanții cumulative

Verificăm că T1 ⊆ T2 ⊆ T3 ⊆ T4 ⊆ T5 pentru lesion_id.

In [10]:
# Check: fiecare tier e superset al celui anterior (la nivel de leziune)
print("Verificare cumulativitate (T_n ⊆ T_{n+1}):\n")

all_cumulative = True
for i in range(1, 5):
    lids_smaller = set(tiers[i]['lesion_id'])
    lids_larger = set(tiers[i+1]['lesion_id'])
    is_subset = lids_smaller.issubset(lids_larger)
    diff = lids_smaller - lids_larger

    status = '[OK]' if is_subset else '[FAIL]'
    print(f"  {status} T{i} ⊆ T{i+1}: {is_subset} (leziuni în T{i} dar nu în T{i+1}: {len(diff)})")
    if not is_subset:
        all_cumulative = False

assert all_cumulative, "Cumulativitate violată!"
print("\n[OK] Toate tier-urile sunt cumulative.")

Verificare cumulativitate (T_n ⊆ T_{n+1}):

  [OK] T1 ⊆ T2: True (leziuni în T1 dar nu în T2: 0)
  [OK] T2 ⊆ T3: True (leziuni în T2 dar nu în T3: 0)
  [OK] T3 ⊆ T4: True (leziuni în T3 dar nu în T4: 0)
  [OK] T4 ⊆ T5: True (leziuni în T4 dar nu în T5: 0)

[OK] Toate tier-urile sunt cumulative.


In [11]:
# Check: T5 = tot train_pool
t5_lids = set(tiers[5]['lesion_id'])
pool_lids = set(train_pool['lesion_id'])

assert t5_lids == pool_lids, f"T5 ({len(t5_lids)}) != train_pool ({len(pool_lids)})"
print(f"[OK] T5 conține toate cele {len(pool_lids):,} leziuni din train_pool.")

[OK] T5 conține toate cele 11,144 leziuni din train_pool.


In [12]:
# Check: T1 = doar HAM10000
t1_sources = set(tiers[1]['source'].unique())
assert t1_sources == {ANCHOR_SOURCE}, f"T1 conține surse: {t1_sources}"
print(f"[OK] T1 conține DOAR {ANCHOR_SOURCE} (verificat prin coloana 'source').")

[OK] T1 conține DOAR HAM10000 (verificat prin coloana 'source').


## 8. Distribuții per tier

In [13]:
# Distribuție clase per tier
class_dist_per_tier = pd.DataFrame({
    f'T{i}': tiers[i]['dx'].value_counts() for i in range(1, 6)
}).fillna(0).astype(int)
class_dist_per_tier.loc['TOTAL'] = class_dist_per_tier.sum()

print("Distribuție clase per tier (numere absolute):")
class_dist_per_tier

Distribuție clase per tier (numere absolute):


,T1,T2,T3,T4,T5
dx,,,,,
akiec,268,502,720,946,1192
bcc,396,929,1495,2077,2661
bkl,888,1177,1504,1836,2128
df,93,118,138,164,184
mel,910,1600,2296,3062,3628
nv,5378,6536,7750,9022,10297
vasc,109,134,158,193,209
TOTAL,8042,10996,14061,17300,20299


In [14]:
# Distribuție clase ca procente per tier
class_dist_pct = pd.DataFrame({
    f'T{i} (%)': (tiers[i]['dx'].value_counts(normalize=True) * 100).round(2)
    for i in range(1, 6)
}).fillna(0)

print("Distribuție clase per tier (procente):")
class_dist_pct

Distribuție clase per tier (procente):


,T1 (%),T2 (%),T3 (%),T4 (%),T5 (%)
dx,,,,,
akiec,3.33,4.57,5.12,5.47,5.87
bcc,4.92,8.45,10.63,12.01,13.11
bkl,11.04,10.70,10.70,10.61,10.48
df,1.16,1.07,0.98,0.95,0.91
mel,11.32,14.55,16.33,17.70,17.87
nv,66.87,59.44,55.12,52.15,50.73
vasc,1.36,1.22,1.12,1.12,1.03


In [15]:
# Distribuție surse per tier
source_dist = pd.DataFrame({
    f'T{i}': tiers[i]['source'].value_counts() for i in range(1, 6)
}).fillna(0).astype(int)
source_dist.loc['TOTAL'] = source_dist.sum()

print("Distribuție surse per tier:")
source_dist

Distribuție surse per tier:


,T1,T2,T3,T4,T5
source,,,,,
BCN_20000,0,2380,4874,7545,9969
HAM10000,8042,8042,8042,8042,8042
MSK,0,159,319,478,651
other,0,415,826,1235,1637
TOTAL,8042,10996,14061,17300,20299


## 9. Salvare CSV-uri per tier

In [16]:
for i in range(1, 6):
    tier_path = os.path.join(OUTPUT_DIR, f'tier{i}_train.csv')
    tiers[i].to_csv(tier_path, index=False)
    print(f"[SAVED] {tier_path}  ({len(tiers[i]):,} imagini)")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier1_train.csv  (8,042 imagini)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier2_train.csv  (10,996 imagini)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier3_train.csv  (14,061 imagini)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier4_train.csv  (17,300 imagini)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier5_train.csv  (20,299 imagini)


## 10. Generare raport markdown

In [17]:
def generate_report(tiers, output_path):
    lines = []
    lines.append("# PASUL 2 — Raport tier-uri")
    lines.append("")
    lines.append(f"**Data generării:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"**Random seed:** {RANDOM_SEED}")
    lines.append("")

    lines.append("## 1. Configurare")
    lines.append("")
    lines.append(f"- **Anchor source:** {ANCHOR_SOURCE} (apare 100% în toate tier-urile)")
    lines.append(f"- **Tier percentages (non-anchor):** {[f'{int(p*100)}%' for p in TIER_PERCENTAGES]}")
    lines.append(f"- **Strategy:** cumulativ (T_n ⊆ T_{{n+1}})")
    lines.append(f"- **Sample-ul non-anchor:** lesion-level, stratificat pe clasă")
    lines.append("")

    lines.append("## 2. Mărimi tier-uri")
    lines.append("")
    lines.append("| Tier | % non-anchor | Imagini | Leziuni |")
    lines.append("|------|--------------|---------|---------|")
    for i in range(1, 6):
        pct = TIER_PERCENTAGES[i-1]
        n_img = len(tiers[i])
        n_les = tiers[i]['lesion_id'].nunique()
        lines.append(f"| T{i} | {int(pct*100)}% | {n_img:,} | {n_les:,} |")
    lines.append("")

    lines.append("## 3. Distribuție clase per tier")
    lines.append("")
    class_dist = pd.DataFrame({
        f'T{i}': tiers[i]['dx'].value_counts() for i in range(1, 6)
    }).fillna(0).astype(int)
    class_dist.loc['TOTAL'] = class_dist.sum()
    lines.append(class_dist.to_markdown())
    lines.append("")

    lines.append("## 4. Distribuție clase procentual per tier")
    lines.append("")
    class_pct = pd.DataFrame({
        f'T{i} (%)': (tiers[i]['dx'].value_counts(normalize=True) * 100).round(2)
        for i in range(1, 6)
    }).fillna(0)
    lines.append(class_pct.to_markdown())
    lines.append("")

    lines.append("## 5. Distribuție surse per tier")
    lines.append("")
    src_dist = pd.DataFrame({
        f'T{i}': tiers[i]['source'].value_counts() for i in range(1, 6)
    }).fillna(0).astype(int)
    src_dist.loc['TOTAL'] = src_dist.sum()
    lines.append(src_dist.to_markdown())
    lines.append("")

    lines.append("## 6. Verificări de garanții")
    lines.append("")
    lines.append("- [x] T1 ⊆ T2 ⊆ T3 ⊆ T4 ⊆ T5 (cumulativitate)")
    lines.append("- [x] T1 conține doar leziuni HAM10000")
    lines.append("- [x] T5 conține toate leziunile din train_pool")
    lines.append("- [x] Sample-ul non-anchor e stratificat pe clasă (DF reprezentat în fiecare tier)")
    lines.append("")

    lines.append("## 7. Decizii metodologice")
    lines.append("")
    lines.append("**De ce cumulativ?**")
    lines.append("")
    lines.append("Pentru a izola efectul *cantității* de date asupra performanței. Cu tier-uri ")
    lines.append("disjuncte, diferențele între T_n și T_{n+1} ar fi confundate cu varianța dintre ")
    lines.append("sample-uri diferite. Cumulativ garantează că trecerea T1 → T2 → ... → T5 ")
    lines.append("**doar adaugă** leziuni, nu schimbă cele existente.")
    lines.append("")
    lines.append("**De ce HAM10000 ca anchor?**")
    lines.append("")
    lines.append("HAM10000 e cel mai studiat dataset dermoscopic și replică direct setup-ul ")
    lines.append("Nawaz et al. (2025). T1 reproduce baseline-ul replicării, iar T2-T5 extind ")
    lines.append("progresiv cu date din ISIC 2019 (BCN_20000, MSK, alte surse).")
    lines.append("")
    lines.append("**De ce sample lesion-level?**")
    lines.append("")
    lines.append("Pentru consistență cu strategia anti-leakage din PASUL 1. Sample la nivel de ")
    lines.append("imagine ar putea fragmenta o leziune (ex: 4 imagini ale leziunii X în T2, ")
    lines.append("celelalte 2 în T3), introducând leakage subtil între tier-uri.")
    lines.append("")
    lines.append("**De ce stratificat pe clasă?**")
    lines.append("")
    lines.append("Pentru a garanta că clasele minoritare (DF, VASC) sunt reprezentate în ")
    lines.append("fiecare tier proporțional cu mărimea lor totală. Fără stratificare, un sample ")
    lines.append("de 25% ar putea, prin hazard, să excludă complet DF din T2.")
    lines.append("")

    lines.append("## 8. Output-uri")
    lines.append("")
    for i in range(1, 6):
        lines.append(f"- `tier{i}_train.csv` — {len(tiers[i]):,} imagini")
    lines.append("")

    lines.append("## 9. Conexiune cu pașii următori")
    lines.append("")
    lines.append("- **PASUL 3:** Pentru fiecare tier, calculăm planul de oversampling pe ")
    lines.append("  clasele minoritare (cu cap la 5×) și salvăm `tier{i}_train_oversampled.csv`.")
    lines.append("- **PASUL 4:** Generăm TFRecords pentru fiecare tier.")
    lines.append("- **PASUL 5:** Antrenăm 4 arhitecturi × 5 tier-uri = 20 modele, ")
    lines.append("  evaluate pe **același val_fixed și test_fixed**.")
    lines.append("")

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


report_path = os.path.join(OUTPUT_DIR, 'step2_report.md')
generate_report(tiers, report_path)
print(f"[SAVED] {report_path}")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step2/step2_report.md


## 11. Checklist de validare

Verifică în output-uri:

- [x] T1 are doar `source = HAM10000`
- [x] T5 are tot train_pool (același număr ca în PASUL 1)
- [X] T1 ⊆ T2 ⊆ T3 ⊆ T4 ⊆ T5 (toate sanity checks `[OK]`)
- [x] DF (clasa minoritară) e prezent în toate cele 5 tier-uri
- [x] La 25%, prefix-ul conține ~25% din fiecare clasă (cu eroare ±2%)

---

## Conexiune cu PASUL 3

`tier{i}_train.csv` va fi input pentru:
- **PASUL 3:** Calculăm offline oversampling plan (per clasă, cap × 5)
- **PASUL 4:** Generăm TFRecords (raw images + aug_id pentru cele oversampled)

Val și test rămân `val_fixed.csv` și `test_fixed.csv` din PASUL 1, neschimbate.